# [Nexia] GenImage 1.0 — ИИ на Google серверах

Google Colab + NVIDIA T4 + ComfyUI + кастомный Gradio UI.

- Конфигурация моделей хранится в `MyDrive/AI_Image_Studio/config.json`.
- Anime: WAI-illustrious-SDXL v17.
- Realistic: RealVisXL V5.0 (SDXL).
- Модели и исходники ComfyUI сохраняются на Google Drive; активная рабочая копия запускается с локального диска Colab.
- PNG сохраняются в `MyDrive/AI_Image_Studio/images/`.
- Источники моделей — Civitai. Hugging Face не используется.

**Перезапуск всей сессии:** если изменился набор зависимостей ComfyUI, ячейка подготовки вызывает `google.colab.runtime.restart()`. Это перезапускает всю сессию Colab, включая Python kernel и процессы, а не только ComfyUI. После перезапуска нужно повторно выполнить ячейки сверху вниз — Colab не продолжает прерванную последовательность автоматически.

При неизменных зависимостях полный перезапуск не выполняется. Доступность GPU и длительность сессии Colab не гарантируются.


## Ячейка 1 — Подготовка среды, установка ComfyUI и полное завершение Colab VM при изменении зависимостей


In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, time, json, shutil, subprocess, requests, random, uuid, io, urllib.parse, hashlib
import torch

# Google Drive — постоянное хранилище
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/AI_Image_Studio")
DRIVE_MODELS = DRIVE_ROOT / "models" / "checkpoints"
DRIVE_IMAGES = DRIVE_ROOT / "images"
DRIVE_COMFY = DRIVE_ROOT / "ComfyUI"
DRIVE_CACHE = DRIVE_ROOT / "pip_cache"
CONFIG_PATH = DRIVE_ROOT / "config.json"
for folder in [DRIVE_ROOT, DRIVE_MODELS, DRIVE_IMAGES, DRIVE_COMFY, DRIVE_CACHE]:
    folder.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available(), "Включите GPU: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), "GB")

# Создаём конфиг только при первом запуске. Пользователь может редактировать его на Google Drive.
DEFAULT_CONFIG = {
  "schema_version": 1,
  "models": [
    {
      "id": "wai_illustrious_v17",
      "name": "WAI Illustrious SDXL v17 · Anime",
      "type": "anime",
      "source": "civitai",
      "version_id": 2883731,
      "filename": "waiIllustriousSDXL_v170.safetensors",
      "enabled": True,
      "default": True,
      "sampler": "euler_ancestral",
      "scheduler": "normal",
      "clip_skip": 2,
      "default_steps": 30,
      "default_cfg": 7
    },
    {
      "id": "realvisxl_v5",
      "name": "RealVisXL V5.0 · Realistic",
      "type": "realistic",
      "source": "civitai",
      "version_id": 789646,
      "filename": "RealVisXL_V5.0.safetensors",
      "enabled": True,
      "default": False,
      "sampler": "dpmpp_2m",
      "scheduler": "karras",
      "clip_skip": 1,
      "default_steps": 30,
      "default_cfg": 5.5
    }
  ]
}
if not CONFIG_PATH.exists():
    CONFIG_PATH.write_text(json.dumps(DEFAULT_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Создан config.json:", CONFIG_PATH)
else:
    print("Используется существующий конфиг:", CONFIG_PATH)

# Проверяем конфигурацию до скачивания файлов.
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
assert isinstance(CONFIG.get("models"), list) and CONFIG["models"], "В config.json отсутствует непустой список models"
seen_ids = set()
for item in CONFIG["models"]:
    for key in ["id", "name", "version_id", "filename"]:
        assert item.get(key), f"У модели отсутствует обязательное поле: {key}"
    assert item["id"] not in seen_ids, f"Повторяющийся model id: {item['id']}"
    seen_ids.add(item["id"])
    assert Path(item["filename"]).name == item["filename"], "filename должен быть именем файла без путей"

# Постоянный исходный код ComfyUI на Drive, рабочая копия — на локальном диске Colab.
COMFY_DRIVE = DRIVE_COMFY
COMFY = Path("/content/ComfyUI")
if not (COMFY_DRIVE / ".git").exists():
    print("Первичная установка ComfyUI на Google Drive...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/comfyanonymous/ComfyUI.git", str(COMFY_DRIVE)], check=True)
else:
    print("Исходники ComfyUI найдены на Google Drive.")

if COMFY.exists():
    shutil.rmtree(COMFY)
shutil.copytree(COMFY_DRIVE, COMFY, ignore=shutil.ignore_patterns(".git", "__pycache__", "*.pyc"))

# pip cache на Drive ускоряет повторную установку пакетов между Colab-сессиями.
os.environ["PIP_CACHE_DIR"] = str(DRIVE_CACHE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(COMFY / "requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=True)
print("ComfyUI подготовлен. Исходники:", COMFY_DRIVE)
print("Рабочая копия:", COMFY)
print("Pip cache:", DRIVE_CACHE)
print("Конфиг моделей:", CONFIG_PATH)

# Полное завершение текущей Colab VM при изменении набора зависимостей.
# runtime.restart() отсутствует в google.colab.runtime.
# runtime.unassign() — поддерживаемый Colab API: завершает назначенную VM
# и отключает notebook. После этого пользователь подключается к новой VM
# кнопкой Connect и повторно запускает ячейки сверху вниз.
requirements_hash = hashlib.sha256((COMFY / "requirements.txt").read_bytes()).hexdigest()
restart_signature = hashlib.sha256((requirements_hash + "|gradio").encode("utf-8")).hexdigest()
RESTART_MARKER = DRIVE_ROOT / ".runtime_restart_signature"
previous_signature = RESTART_MARKER.read_text(encoding="utf-8").strip() if RESTART_MARKER.exists() else ""

if previous_signature != restart_signature:
    # Маркер записываем ДО отключения, чтобы после подключения не зациклиться.
    RESTART_MARKER.write_text(restart_signature, encoding="utf-8")
    print("\nОбнаружена новая/изменённая конфигурация зависимостей.")
    print("Завершаю текущую VM Google Colab для чистого запуска.")
    print("После отключения нажмите Connect и повторно выполните ячейки сверху вниз.")
    from google.colab import runtime
    runtime.unassign()
else:
    print("Изменений зависимостей нет: завершение Colab VM не требуется.")


## Ячейка 2 — Менеджер моделей из config.json

Скрипт проверяет наличие файлов на Google Drive, проверяет доступность версий на Civitai, загружает отсутствующие модели один раз и копирует выбранные checkpoint в локальную папку ComfyUI. Для добавления модели редактируйте `config.json` — формат описан в самом файле.

In [ ]:
import requests, os, json, time, shutil, hashlib
from pathlib import Path

CIVITAI_TOKEN = ""  # Необязательно. Не публикуйте ноутбук с личным токеном.

def civitai_headers():
    headers = {"User-Agent": "AI-Image-Studio-Colab/2.0"}
    if CIVITAI_TOKEN.strip():
        headers["Authorization"] = f"Bearer {CIVITAI_TOKEN.strip()}"
    return headers

def get_civitai_version(version_id):
    url = f"https://civitai.com/api/v1/model-versions/{int(version_id)}"
    response = requests.get(url, headers=civitai_headers(), timeout=60)
    response.raise_for_status()
    data = response.json()
    if data.get("availability") not in (None, "Public"):
        raise RuntimeError(f"Версия Civitai {version_id} недоступна публично: {data.get('availability')}")
    files = data.get("files", [])
    safe = [f for f in files if f.get("type") == "Model" and (f.get("metadata", {}).get("format") == "SafeTensor" or f.get("name", "").lower().endswith(".safetensors"))]
    if not safe:
        raise RuntimeError(f"Для Civitai version_id={version_id} не найден публичный SafeTensor checkpoint")
    primary = next((f for f in safe if f.get("primary")), safe[0])
    if primary.get("pickleScanResult") == "Dangerous":
        raise RuntimeError("Civitai пометил файл как потенциально опасный; загрузка остановлена.")
    return data, primary

def download_civitai_version(version_id, destination):
    version, file_info = get_civitai_version(version_id)  # Проверка доступности и метаданных перед загрузкой
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    expected_size = int(float(file_info.get("sizeKB", 0)) * 1024) if file_info.get("sizeKB") else 0
    if destination.exists() and destination.stat().st_size > 100_000_000 and (not expected_size or destination.stat().st_size == expected_size):
        print("Файл уже существует и размер совпадает:", destination.name)
        return destination
    url = file_info.get("downloadUrl") or f"https://civitai.com/api/download/models/{int(version_id)}"
    tmp = destination.with_suffix(destination.suffix + ".part")
    done = 0
    with requests.get(url, headers=civitai_headers(), stream=True, allow_redirects=True, timeout=(30, 600)) as response:
        response.raise_for_status()
        with open(tmp, "wb") as output:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    output.write(chunk)
                    done += len(chunk)
                    if expected_size:
                        print(f"\r{done / 1024**3:.2f}/{expected_size / 1024**3:.2f} GB", end="")
    if done < 100_000_000 or (expected_size and done != expected_size):
        tmp.unlink(missing_ok=True)
        raise RuntimeError(f"Размер скачанного файла не прошёл проверку: {done} bytes; ожидалось {expected_size}")
    tmp.replace(destination)
    print("\nСкачано:", destination)
    return destination

def ensure_model_available(model_config):
    drive_path = DRIVE_MODELS / model_config["filename"]
    local_path = Path(COMFY) / "models" / "checkpoints" / model_config["filename"]
    drive_path.parent.mkdir(parents=True, exist_ok=True)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if not drive_path.exists() or drive_path.stat().st_size < 100_000_000:
        print(f"\n[{model_config['name']}] На Google Drive нет полной модели — проверяю Civitai и скачиваю...")
        download_civitai_version(model_config["version_id"], drive_path)
    else:
        # Всё равно проверяем, что Civitai-версия существует и публична.
        get_civitai_version(model_config["version_id"])
        print(f"[{model_config['name']}] Найдена на Google Drive.")
    if not local_path.exists() or local_path.stat().st_size != drive_path.stat().st_size:
        print(f"Копирую {model_config['filename']} с Drive на локальный диск Colab...")
        shutil.copy2(drive_path, local_path)
    return local_path

ENABLED_MODELS = [m for m in CONFIG["models"] if m.get("enabled", True)]
if not ENABLED_MODELS:
    raise RuntimeError("В config.json нет включённых моделей.")
DEFAULT_MODEL = next((m for m in ENABLED_MODELS if m.get("default")), ENABLED_MODELS[0])
MODEL_CONFIG_BY_NAME = {m["name"]: m for m in ENABLED_MODELS}
MODEL_CHECKPOINTS = {}
for model_config in ENABLED_MODELS:
    try:
        local_file = ensure_model_available(model_config)
        MODEL_CHECKPOINTS[model_config["name"]] = local_file.name
        print("ГОТОВО:", model_config["name"], "→", local_file.name)
    except Exception as exc:
        print(f"ПРЕДУПРЕЖДЕНИЕ: модель '{model_config['name']}' пропущена: {type(exc).__name__}: {exc}")

if not MODEL_CHECKPOINTS:
    raise RuntimeError("Не удалось подготовить ни одной модели. Проверьте config.json и доступ к Civitai.")
AVAILABLE_MODEL_NAMES = [name for name in MODEL_CONFIG_BY_NAME if name in MODEL_CHECKPOINTS]
DEFAULT_MODEL_NAME = DEFAULT_MODEL["name"] if DEFAULT_MODEL["name"] in MODEL_CHECKPOINTS else AVAILABLE_MODEL_NAMES[0]
print("\nДоступные модели:", AVAILABLE_MODEL_NAMES)
print("Модель по умолчанию:", DEFAULT_MODEL_NAME)

## Ячейка 3 — Запуск ComfyUI с автоматическим восстановлением backend

In [ ]:
import subprocess, sys, time, requests
from pathlib import Path

COMFY = Path("/content/ComfyUI")
COMFY_URL = "http://127.0.0.1:8188"
comfy_proc = None
backend_log = None

def start_comfy_backend():
    global comfy_proc, backend_log
    if comfy_proc is not None and comfy_proc.poll() is None:
        try:
            if requests.get(COMFY_URL + "/system_stats", timeout=3).ok:
                return
        except Exception:
            pass
        comfy_proc.terminate()
        try:
            comfy_proc.wait(timeout=10)
        except Exception:
            comfy_proc.kill()
    backend_log = open("/content/comfyui.log", "a", buffering=1)
    comfy_proc = subprocess.Popen(
        [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188", "--lowvram"],
        cwd=str(COMFY), stdout=backend_log, stderr=subprocess.STDOUT
    )
    for _ in range(150):
        if comfy_proc.poll() is not None:
            break
        try:
            if requests.get(COMFY_URL + "/system_stats", timeout=2).ok:
                print("ComfyUI backend запущен (PID", comfy_proc.pid, ").")
                return
        except Exception:
            time.sleep(2)
    log = Path("/content/comfyui.log").read_text(errors="replace")[-8000:]
    raise RuntimeError("ComfyUI не запустился. Последние строки лога:\n" + log)

start_comfy_backend()

## Ячейка 4 — Кастомный интерфейс AI Image Studio

Полностью переработанный тёмный интерфейс на собственных CSS-стилях и компоновке. Внутри используется Gradio только как транспорт событий и публикация страницы; стандартная раскладка заменена кастомной.


In [ ]:
import gradio as gr
from PIL import Image
import traceback
import os
import io
import time
import json
import random
import requests
from pathlib import Path

COMFY = "/content/ComfyUI"
COMFY_URL = "http://127.0.0.1:8188"
DRIVE_ROOT = Path("/content/drive/MyDrive/AI_Image_Studio")
OUTPUT_DIR = DRIVE_ROOT / "images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def queue_workflow(workflow):
    try:
        response = requests.post(
            f"{COMFY_URL}/prompt",
            json={"prompt": workflow},
            timeout=90
        )
    except requests.RequestException as exc:
        raise RuntimeError(f"ComfyUI недоступен: {type(exc).__name__}: {exc}")

    if not response.ok:
        try:
            details = json.dumps(response.json(), ensure_ascii=False, indent=2)
        except Exception:
            details = response.text[:5000]
        raise RuntimeError(f"ComfyUI HTTP {response.status_code}:\n{details}")

    payload = response.json()
    prompt_id = payload.get("prompt_id")
    if not prompt_id:
        raise RuntimeError(f"ComfyUI не вернул prompt_id: {payload}")
    return prompt_id


def wait_image(prompt_id, timeout=1800):
    started = time.time()
    while time.time() - started < timeout:
        response = requests.get(f"{COMFY_URL}/history/{prompt_id}", timeout=30)
        response.raise_for_status()
        history = response.json()
        entry = history.get(prompt_id)

        if entry:
            status = entry.get("status", {})
            messages = status.get("messages", [])
            errors = [
                message for message in messages
                if isinstance(message, (list, tuple))
                and message and message[0] == "execution_error"
            ]
            if status.get("status_str") == "error" or errors:
                raise RuntimeError(json.dumps(
                    errors or status, ensure_ascii=False, indent=2, default=str
                ))

            if status.get("completed") or status.get("status_str") == "success":
                images = [
                    image_data
                    for node in entry.get("outputs", {}).values()
                    for image_data in node.get("images", [])
                ]
                if not images:
                    raise RuntimeError(
                        "ComfyUI завершил workflow, но не вернул файл изображения."
                    )

                item = images[0]
                image_response = requests.get(
                    f"{COMFY_URL}/view",
                    params={
                        "filename": item["filename"],
                        "subfolder": item.get("subfolder", ""),
                        "type": item.get("type", "output")
                    },
                    timeout=180
                )
                image_response.raise_for_status()
                image = Image.open(io.BytesIO(image_response.content))
                image.load()
                return image.convert("RGB")

        time.sleep(1.5)

    raise TimeoutError(f"Генерация не завершилась за {timeout} секунд.")


def make_workflow(model_config, checkpoint_name, positive, negative, width, height, steps, cfg, seed):
    width = max(512, min(1536, int(width) // 64 * 64))
    height = max(512, min(1536, int(height) // 64 * 64))

    return {
        "1": {
            "class_type": "CheckpointLoaderSimple",
            "inputs": {"ckpt_name": checkpoint_name}
        },
        "8": {
            "class_type": "CLIPSetLastLayer",
            "inputs": {"clip": ["1", 1], "stop_at_clip_layer": -int(model_config.get("clip_skip", 1))}
        },
        "2": {
            "class_type": "CLIPTextEncode",
            "inputs": {"text": positive, "clip": ["8", 0]}
        },
        "3": {
            "class_type": "CLIPTextEncode",
            "inputs": {
                "text": negative or "low quality, blurry, bad anatomy, watermark",
                "clip": ["8", 0]
            }
        },
        "4": {
            "class_type": "EmptyLatentImage",
            "inputs": {
                "width": width,
                "height": height,
                "batch_size": 1
            }
        },
        "5": {
            "class_type": "KSampler",
            "inputs": {
                "seed": int(seed),
                "steps": int(steps),
                "cfg": float(cfg),
                "sampler_name": model_config.get("sampler", "dpmpp_2m"),
                "scheduler": model_config.get("scheduler", "karras"),
                "denoise": 1.0,
                "model": ["1", 0],
                "positive": ["2", 0],
                "negative": ["3", 0],
                "latent_image": ["4", 0]
            }
        },
        "6": {
            "class_type": "VAEDecode",
            "inputs": {"samples": ["5", 0], "vae": ["1", 2]}
        },
        "7": {
            "class_type": "SaveImage",
            "inputs": {
                "filename_prefix": "AI_Image_Studio",
                "images": ["6", 0]
            }
        }
    }


def generate(model_name, positive, negative, width, height, steps, cfg, seed):
    try:
        if not positive or not positive.strip():
            return None, None, "Введите описание изображения."
        if model_name not in MODEL_CONFIG_BY_NAME or model_name not in MODEL_CHECKPOINTS:
            raise RuntimeError("Выбранная модель недоступна. Проверьте config.json и выполните ячейку менеджера моделей.")
        # Автоматически восстанавливаем ComfyUI, если backend завершился.
        start_comfy_backend()
        model_config = MODEL_CONFIG_BY_NAME[model_name]
        checkpoint_name = MODEL_CHECKPOINTS[model_name]
        if seed is None or int(seed) < 0:
            seed = random.randint(0, 2**63 - 1)
        else:
            seed = int(seed)
        checkpoint_path = Path(COMFY) / "models" / "checkpoints" / checkpoint_name
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"Модель не найдена: {checkpoint_path}. Повторно выполните ячейку менеджера моделей.")
        workflow = make_workflow(model_config, checkpoint_name, positive, negative, width, height, steps, cfg, seed)
        prompt_id = queue_workflow(workflow)
        result = wait_image(prompt_id)
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        output_path = str(OUTPUT_DIR / f"AI_Image_{prompt_id}.png")
        result.save(output_path, format="PNG")
        info = f"ГОТОВО · {model_config['name']} · {result.width}×{result.height} · seed {seed}"
        return result, output_path, info
    except Exception as exc:
        traceback.print_exc()
        return None, None, f"ОШИБКА ГЕНЕРАЦИИ\n{type(exc).__name__}: {exc}"


CUSTOM_CSS = r"""
:root {
  --studio-bg: #090b10;
  --studio-panel: #11151e;
  --studio-panel-2: #171d29;
  --studio-border: rgba(255,255,255,.09);
  --studio-text: #edf1f8;
  --studio-muted: #8b96a9;
  --studio-accent: #9b8cff;
}
.gradio-container {
  max-width: 1450px !important;
  margin: 0 auto !important;
  background:
    radial-gradient(ellipse at 12% 0%, rgba(116,91,255,.14), transparent 35%),
    radial-gradient(ellipse at 92% 8%, rgba(40,177,190,.10), transparent 28%),
    var(--studio-bg) !important;
  color: var(--studio-text) !important;
  font-family: Inter, ui-sans-serif, system-ui, sans-serif !important;
}
#studio-shell { gap: 18px; }
#studio-header {
  padding: 26px 30px;
  border: 1px solid var(--studio-border);
  border-radius: 22px;
  background: linear-gradient(120deg, rgba(27,33,48,.97), rgba(15,20,30,.94));
  box-shadow: 0 18px 60px rgba(0,0,0,.22);
}
#studio-header h1 {
  color: #fff; font-size: 30px; font-weight: 750;
  letter-spacing: -.8px; margin: 0 0 7px 0;
}
#studio-header p { color: var(--studio-muted); margin: 0; font-size: 14px; }
#studio-header .studio-kicker {
  color: #b6aaff; font-size: 11px; font-weight: 800;
  letter-spacing: 2px; text-transform: uppercase; margin-bottom: 10px;
}
.studio-panel {
  background: rgba(17,21,30,.94) !important;
  border: 1px solid var(--studio-border) !important;
  border-radius: 20px !important;
  padding: 20px !important;
}
.studio-section-title {
  color: #f1f3fa; font-size: 13px; font-weight: 800;
  letter-spacing: 1.2px; text-transform: uppercase;
  margin: 0 0 12px 0;
}
.studio-hint { color: var(--studio-muted); font-size: 12px; }
#prompt-positive textarea, #prompt-negative textarea {
  background: #0c1018 !important; border: 1px solid #293143 !important;
  color: #edf1f8 !important; border-radius: 13px !important;
  line-height: 1.55 !important;
}
#prompt-positive textarea { min-height: 150px !important; }
#prompt-negative textarea { min-height: 92px !important; }
#studio-shell input, #studio-shell button, #studio-shell .wrap {
  border-radius: 12px !important;
}
#generate-btn {
  background: linear-gradient(100deg, #7967ee, #a17af7) !important;
  border: 0 !important; color: white !important;
  min-height: 54px !important; font-weight: 800 !important;
  font-size: 15px !important; box-shadow: 0 9px 28px rgba(130,103,238,.24);
}
#generate-btn:hover { filter: brightness(1.08); }
#result-image {
  border: 1px solid var(--studio-border) !important;
  border-radius: 16px !important;
  overflow: hidden !important;
  background: #090c12 !important;
}
#status-box textarea {
  background: #0c1018 !important; color: #b9c5d8 !important;
  border: 1px solid #293143 !important; font-family: ui-monospace, monospace !important;
  font-size: 12px !important;
}
#download-file {
  border: 1px solid var(--studio-border) !important;
  background: #151b27 !important;
}
footer { display: none !important; }
"""

with gr.Blocks(
    title="AI Image Studio",
    theme=gr.themes.Base(
        primary_hue="violet",
        secondary_hue="slate",
        neutral_hue="slate"
    ),
    css=CUSTOM_CSS,
    elem_id="studio-shell"
) as app:

    gr.HTML("""
    <div id="studio-header">
      <div class="studio-kicker">LOCAL INFERENCE · COMFYUI BACKEND</div>
      <h1>AI Image Studio</h1>
      <p>Anime и photorealistic SDXL · модели управляются через config.json</p>
    </div>
    """)

    with gr.Row(equal_height=False):
        with gr.Column(scale=5, min_width=360):
            with gr.Group(elem_classes=["studio-panel"]):
                gr.HTML('<div class="studio-section-title">01 / Модель и промпт</div>')
                model = gr.Dropdown(
                    choices=AVAILABLE_MODEL_NAMES,
                    value=DEFAULT_MODEL_NAME,
                    label="Выбери checkpoint"
                )
                positive = gr.Textbox(
                    label="Positive prompt",
                    placeholder="Опиши сцену, персонажа, стиль, освещение, композицию...",
                    lines=6,
                    elem_id="prompt-positive"
                )
                negative = gr.Textbox(
                    label="Negative prompt",
                    placeholder="Что исключить из изображения...",
                    lines=3,
                    elem_id="prompt-negative"
                )

            with gr.Group(elem_classes=["studio-panel"]):
                gr.HTML('<div class="studio-section-title">02 / Параметры генерации</div>')
                with gr.Row():
                    width = gr.Dropdown(
                        choices=[512, 576, 640, 704, 768, 832, 896, 960, 1024, 1088, 1152, 1216, 1280, 1344, 1408, 1472, 1536],
                        value=1024,
                        label="Ширина"
                    )
                    height = gr.Dropdown(
                        choices=[512, 576, 640, 704, 768, 832, 896, 960, 1024, 1088, 1152, 1216, 1280, 1344, 1408, 1472, 1536],
                        value=1344,
                        label="Высота"
                    )
                with gr.Row():
                    steps = gr.Slider(10, 40, value=30, step=1, label="Steps")
                    cfg = gr.Slider(1, 12, value=7, step=0.5, label="CFG")
                seed = gr.Number(
                    value=-1,
                    precision=0,
                    label="Seed  ·  -1 = случайный"
                )
                generate_button = gr.Button(
                    "✦  Сгенерировать изображение",
                    variant="primary",
                    elem_id="generate-btn"
                )
                status = gr.Textbox(
                    label="Состояние",
                    value="ОЖИДАНИЕ · Заполни prompt и нажми «Сгенерировать»",
                    lines=3,
                    interactive=False,
                    elem_id="status-box"
                )

        with gr.Column(scale=6, min_width=360):
            with gr.Group(elem_classes=["studio-panel"]):
                gr.HTML('<div class="studio-section-title">03 / Результат</div>')
                preview = gr.Image(
                    label="Generated image",
                    type="pil",
                    height=570,
                    elem_id="result-image"
                )
                download = gr.File(
                    label="PNG · Скачать изображение",
                    elem_id="download-file"
                )
                gr.HTML(
                    '<div class="studio-hint">Изображение сохраняется в PNG. '
                    'Seed можно повторно использовать для воспроизведения результата.</div>'
                )

    generate_button.click(
        fn=generate,
        inputs=[model, positive, negative, width, height, steps, cfg, seed],
        outputs=[preview, download, status],
        api_name="generate",
        show_progress="full"
    )

app.queue(default_concurrency_limit=1)
app.launch(share=True, debug=True, allowed_paths=[str(OUTPUT_DIR)])


## Постоянное хранение и управление моделями

- `MyDrive/AI_Image_Studio/config.json` — список моделей и их параметры. Измените JSON, затем повторно выполните ячейку менеджера моделей и ячейку запуска интерфейса.
- `models/checkpoints/` — постоянные checkpoint-файлы, загружаемые с Civitai только при отсутствии на Drive.
- `ComfyUI/` — исходники ComfyUI на Google Drive. Перед запуском они копируются в `/content/ComfyUI`, чтобы не запускать backend напрямую с Drive.
- `pip_cache/` — кеш pip на Google Drive; пакеты всё равно проверяются/устанавливаются в новой временной среде Colab.
- `images/` — PNG-результаты.

### Добавление новой модели

Добавьте объект в массив `models` файла `config.json`. Обязательные поля: `id`, `name`, `type`, `source`, `version_id`, `filename`. `version_id` — ID конкретной версии checkpoint на Civitai. Поддерживается SafeTensor. Не добавляйте закрытые или неизвестные источники.

### Перезапуск полной сессии Google Colab

После установки Python-зависимостей вычисляется подпись `requirements.txt`. Если она отличается от сохранённой на Google Drive, ноутбук записывает маркер и вызывает `google.colab.runtime.restart()` — это перезапускает всю Colab-сессию, включая Python kernel и процессы. После перезапуска необходимо повторно выполнить ячейки ноутбука сверху вниз: Colab не возобновляет прерванную последовательность ячеек автоматически. При неизменных зависимостях перезапуск не выполняется.

Отдельно перед генерацией проверяется доступность ComfyUI; если backend аварийно завершился, он запускается повторно без перезапуска всей сессии.

### Важные ограничения

- Google Colab может завершить сессию и ограничить доступ к GPU.
- Установка Python-пакетов в новой VM всё ещё требуется; pip cache на Drive может ускорить её, но не гарантирует отсутствие загрузок.
- Для T4 16 GB используйте умеренные разрешения; большие SDXL-кадры могут вызвать CUDA Out of Memory.
- Все модели загружаются из Civitai. Hugging Face не используется.